In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("lab-2") \
    .master("local[11]") \
    .config("spark.sql.shuffle.partitions", 11) \
    .config("spark.eventLog.enabled", "false") \
    .getOrCreate()

In [2]:
s = "country STRING, temperature DOUBLE, event_time STRING"

In [3]:
df = spark.readStream \
    .schema(s) \
    .json("file:///home/itversity/spark/lab2/temperature_input")

In [4]:
from pyspark.sql.functions import col

df = df.select(
    "country",
    "temperature",
    col("event_time").cast("timestamp").alias("event_time")
)

In [5]:
df_watermark = df.withWatermark(
    "event_time",
    "10 minutes"
)

In [6]:
from pyspark.sql.functions import window, avg

df_avg = df_watermark.groupBy("country",window("event_time", "15 minutes")) \
    .agg(avg("temperature").alias("avg_temperature"))


In [7]:
result = df_avg.select(
    "country",
    "window.start",
    "window.end",
    "avg_temperature"
)

In [8]:
file_query = result.writeStream \
    .format("csv") \
    .outputMode("append") \
    .option("path", "file:///home/itversity/spark/lab2/output/temperature_avg") \
    .option("checkpointLocation","checkpoints/files")

In [9]:
q= file_query.start()

In [10]:
spark.sql(""" 
CREATE TABLE average_temperatures (
    country STRING,
    start TIMESTAMP,
    end TIMESTAMP,
    avg_temperature DOUBLE
)
USING parquet
""")

""


In [11]:
hive_query = result.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation","checkpoints/hive") \
    .toTable("average_temperatures")